# SIPTA — Validación: Demografía y Población (Localidad & UPL)
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: CONTROL | **Fase CRISP-DM**: Data Quality & Validation  
**Autoría**: Persona A (Adan Sánchez) & Persona B (Yesid Bello)  
**Fuente**: `data/raw/DEMOGRAFIA/osb_demografia-poblacion-localidad.csv` y `osb_demografia-poblacion-upl.csv`  
**Temporalidad**: **2005 - 2035 (Proyecciones Anuales SDP - DANE, CNPV 2018)**  
**Indicadores Habilitados**: `DEM-001` (Densidad Poblacional), `POB-001..004` (Población por Grupos de Edad)


## 1. Validación de Calidad Técnica con `src/validation/validate_data.py`


In [ ]:
import sys
from pathlib import Path

# Resolver la raíz del proyecto SIPTA
for p in [Path('.').resolve(), Path('.').resolve().parent, Path('.').resolve().parent.parent]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import pandas as pd
from src.validation.validate_data import validate_demografia, inspect_schema, load_raw_demografia

report = validate_demografia()
print("=== REPORTE EJECUTIVO DE VALIDACIÓN ===")
print(f"Dominio: {report['domain']}")
print(f"Temporalidad Oficial: {report['temporalidad']}")
print(f"Total Registros: {report['total_rows']:,}")
print(f"Rango de Años: {report['years_covered'][0]} a {report['years_covered'][-1]}")
print(f"Estado de Calidad: {report['validation_status']}")



## 2. Inspección Detallada de Esquema y Nulos


In [ ]:
df_loc, df_upl = load_raw_demografia()
schema_df = inspect_schema(df_loc)
display(schema_df)



## 3. Demostración de Cálculo de Indicadores (`DEM-001` y `POB-002`)


In [ ]:
# 1. Agregación de población proyectada para el año 2025 por Localidad
col_anio = [c for c in df_loc.columns if 'an' in c.lower() or 'añ' in c.lower()][0]
df_2025 = df_loc[df_loc[col_anio] == 2025].copy()
pob_2025 = df_2025.groupby(['CODIGO_LOCALIDAD', 'NOMBRE_LOCALIDAD'])['POBLACION'].sum().reset_index()

# 2. Población escolar (5 a 17 años) para indicadores educativos (EDU-001 / EDU-003)
pob_escolar = df_2025[df_2025['EDAD'].between(5, 17)].groupby('CODIGO_LOCALIDAD')['POBLACION'].sum().reset_index()
pob_escolar.rename(columns={'POBLACION': 'POBLACION_5_17'}, inplace=True)

df_demo_ind = pd.merge(pob_2025, pob_escolar, on='CODIGO_LOCALIDAD')
df_demo_ind['PCT_POB_ESCOLAR'] = round((df_demo_ind['POBLACION_5_17'] / df_demo_ind['POBLACION']) * 100.0, 2)

print("Demostración de denominadores demográficos por Localidad (2025):")
display(df_demo_ind.head(10))



## 4. Dictamen de Validez
- **Fuente Válida**: Sí. Cubre el 100% de las 20 localidades oficiales sin nulos en `CODIGO_LOCALIDAD` ni `POBLACION`.
- **Uso Metodológico**: Denominador per cápita oficial de todo el sistema SIPTA.
